# 06 — Options Analysis (Deribit DVOL)

Optional phase. Only run if DVOL data covers the event dates.

Questions:
1. Does IV expand around prediction market jump events?
2. Does IV lead prediction markets (like equity options lead earnings)?
3. Is IV compressed at T=0? → Straddle entry attractive

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.lead_lag import compute_ccf

raw_dir  = Path('../data/raw')
proc_dir = Path('../data/processed')

# Load DVOL (daily)
dvol_path = raw_dir / 'deribit' / 'dvol_btc.parquet'
if not dvol_path.exists():
    print('DVOL data not found. Run notebook 01 first.')
else:
    dvol = pd.read_parquet(dvol_path).set_index('timestamp_utc').sort_index()
    print(f'DVOL: {len(dvol)} daily records ({dvol.index.min().date()} to {dvol.index.max().date()})')
    dvol.head()

In [ ]:
if dvol_path.exists():
    # Resample DVOL to daily; compute rolling 7-day mean
    dvol_daily = dvol['volatility']
    dvol_7d_mean = dvol_daily.rolling(7, min_periods=1).mean()
    dvol_deviation = dvol_daily - dvol_7d_mean  # positive = IV elevated

    # Load D1 signals (or whichever passed event study)
    sig_path = proc_dir / 'jump_signals_D1.parquet'
    if sig_path.exists():
        sig_df = pd.read_parquet(sig_path)
        combined = sig_df.stack()
        combined = combined[combined != 0]
        flat_signal = combined.groupby(level=0).apply(lambda x: x.iloc[x.abs().argmax()])

        # Resample signal events to daily
        event_days = flat_signal[flat_signal != 0].resample('1D').apply(
            lambda x: x.iloc[x.abs().argmax()] if len(x) > 0 else 0
        ).fillna(0)

        # 1. IV change around events
        event_times = event_days[event_days != 0].index
        iv_before = dvol_daily.reindex(event_times, method='nearest')
        iv_after  = dvol_daily.shift(-1).reindex(event_times, method='nearest')
        iv_change = iv_after - iv_before

        directions = event_days[event_days != 0]
        pos_events = directions[directions > 0].index
        neg_events = directions[directions < 0].index

        print('=== IV Change Around Jump Events (Day After vs Day Of) ===')
        print(f'Positive jumps (n={len(pos_events)}): mean IV change = {iv_change.reindex(pos_events).mean():.2f} vol pts')
        print(f'Negative jumps (n={len(neg_events)}): mean IV change = {iv_change.reindex(neg_events).mean():.2f} vol pts')

        # 2. IV deviation at event time: is IV compressed before the jump?
        iv_dev_at_event = dvol_deviation.reindex(event_times, method='nearest')
        print(f'\nMean IV deviation at jump time: {iv_dev_at_event.mean():.2f} vol pts')
        print(f'Fraction of events where IV was below 7-day mean: {(iv_dev_at_event < 0).mean():.1%}')

        # 3. CCF: does DVOL lead prediction markets?
        delta_dvol = dvol_daily.diff().dropna()
        delta_prob = event_days.diff().fillna(0)
        aligned = pd.concat([delta_dvol, delta_prob], axis=1).dropna()
        if len(aligned) > 20:
            lags = [-14, -7, -3, -1, 0, 1, 3, 7, 14]  # daily lags
            ccf = compute_ccf(aligned.iloc[:,0], aligned.iloc[:,1], lags=lags)
            print('\nCCF: DVOL vs Prediction Market Jumps (daily lags)')
            print(ccf[['lag_min', 'correlation']].to_string(index=False))
            print('Positive lag = DVOL leads prediction market')

        # Strategy decision
        print('\n=== Strategy Decision ===')
        if iv_dev_at_event.mean() < 0:
            print('IV is typically BELOW 7-day mean at jump time → Straddle may be cheap → Strategy S2 viable')
        else:
            print('IV is typically ABOVE 7-day mean at jump time → Options expensive → Prefer spot (Strategy S1)')